In [ ]:
import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem import AllChem, rdMolDescriptors
from rdkit.Chem.Draw import IPythonConsole

In [ ]:
df = pd.read_csv(r"C:\Users\Dell\PycharmProjects\Molecular-Property-Prediction-Using-GNN-and-Transformer\Dataset\qm8.csv")
df.head()

In [ ]:
print("dataset shape:", df.shape)

In [ ]:
print("\ncolumns:", df.columns)

In [ ]:
# Convert SMILES to RDKit molecule objects
df["mol"] = df["smiles"].apply(Chem.MolFromSmiles)

# Remove invalid molecules
df = df[df["mol"].notnull()]

print("Valid molecules:", len(df))


In [ ]:
from rdkit.Chem import Draw

mol = df["mol"].iloc[0]

Draw.MolToImage(mol, size=(300,300))


In [ ]:
mols = df["mol"].iloc[:12]

Draw.MolsToGridImage(
    mols,
    molsPerRow=4,
    subImgSize=(200,200)
)


In [ ]:
for atom in mol.GetAtoms():
    print("Atom:", atom.GetSymbol(),
          "Degree:", atom.GetDegree(),
          "Hybridization:", atom.GetHybridization())


In [ ]:
for bond in mol.GetBonds():
    print("Bond:", bond.GetBondType())


In [ ]:
def get_descriptors(mol):
    return {
        "MolWt": Descriptors.MolWt(mol),
        "LogP": Descriptors.MolLogP(mol),
        "HBD": Descriptors.NumHDonors(mol),
        "HBA": Descriptors.NumHAcceptors(mol),
        "TPSA": Descriptors.TPSA(mol)
    }

desc = df["mol"].apply(get_descriptors)

desc_df = pd.DataFrame(desc.tolist())
print(desc_df.head())


In [ ]:
import matplotlib.pyplot as plt

desc_df.hist(bins=30, figsize=(12,8))
plt.show()


In [ ]:
import torch

def atom_features(atom):
    return torch.tensor([
        atom.GetAtomicNum(),
        atom.GetDegree(),
        atom.GetFormalCharge(),
        atom.GetHybridization().real,
        atom.GetIsAromatic()
    ], dtype=torch.float)


In [ ]:
def bond_features(bond):
    return torch.tensor([
        bond.GetBondTypeAsDouble(),
        bond.GetIsConjugated(),
        bond.IsInRing()
    ], dtype=torch.float)


In [ ]:
from torch_geometric.data import Data

def mol_to_graph(mol, target):

    node_feats = []
    edge_index = []
    edge_attr = []

    # Nodes
    for atom in mol.GetAtoms():
        node_feats.append(atom_features(atom))

    # Edges
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()

        edge_index.append([i, j])
        edge_index.append([j, i])

        bf = bond_features(bond)
        edge_attr.append(bf)
        edge_attr.append(bf)

    x = torch.stack(node_feats)
    edge_index = torch.tensor(edge_index).t().contiguous()
    edge_attr = torch.stack(edge_attr)

    y = torch.tensor([target], dtype=torch.float)

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)
